# Laboratório Prático #1: Mini Lambda Architecture no Colab
Vamos simular uma Lambda Architecture para detecção de fraude em tempo real.

In [1]:
# --- Célula 1: Setup ---
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("Lab01_Lambda").getOrCreate()
print("Spark pronto para Lambda Architecture!")

Spark pronto para Lambda Architecture!


In [2]:
# --- Célula 2: Batch Layer (Dados Históricos) ---
from pyspark.sql.functions import col, sum as spark_sum
import pandas as pd

# Simular 1 milhão de transações históricas
data_historico = spark.range(0, 1000000).select(
    (col("id") % 1000).alias("user_id"),
    (col("id") % 100).alias("valor"),
    ((col("id") % 10) < 1).cast("int").alias("fraude") # 10% de fraude
)

# Batch View: Agregação por usuário
batch_view = data_historico.groupBy("user_id").agg(
    spark_sum("valor").alias("total_gasto"),
    spark_sum("fraude").alias("total_fraudes")
)

print("Batch View (Histórico):")
batch_view.show(5)

Batch View (Histórico):
+-------+-----------+-------------+
|user_id|total_gasto|total_fraudes|
+-------+-----------+-------------+
|     26|      26000|            0|
|     29|      29000|            0|
|    474|      74000|            0|
|    964|      64000|            0|
|     65|      65000|            0|
+-------+-----------+-------------+
only showing top 5 rows



In [3]:
# --- Célula 3: Speed Layer (Dados em Tempo Real) ---
# Simular 1000 transações que chegaram nos últimos 5 minutos
data_stream = spark.range(0, 1000).select(
    (col("id") % 1000).alias("user_id"),
    (col("id") % 50).alias("valor"),
    ((col("id") % 5) < 1).cast("int").alias("fraude") # 20% de fraude (aumentou!)
)

# Real-time View: Agregação dos dados novos
realtime_view = data_stream.groupBy("user_id").agg(
    spark_sum("valor").alias("total_gasto_rt"),
    spark_sum("fraude").alias("total_fraudes_rt")
)

print("Real-time View (Últimos 5 minutos):")
realtime_view.show(5)

Real-time View (Últimos 5 minutos):
+-------+--------------+----------------+
|user_id|total_gasto_rt|total_fraudes_rt|
+-------+--------------+----------------+
|     26|            26|               0|
|     29|            29|               0|
|    474|            24|               0|
|     65|            15|               1|
|    191|            41|               0|
+-------+--------------+----------------+
only showing top 5 rows



In [4]:
# --- Célula 4: Serving Layer (Combinar Batch + Real-time) ---
# Join das duas views
serving_view = batch_view.join(realtime_view, "user_id", "outer").fillna(0)

# Calcular totais combinados
from pyspark.sql.functions import coalesce, lit

serving_view = serving_view.withColumn(
    "total_gasto_combinado",
    col("total_gasto") + col("total_gasto_rt")
).withColumn(
    "total_fraudes_combinado",
    col("total_fraudes") + col("total_fraudes_rt")
)

print("Serving View (Batch + Real-time):")
serving_view.select("user_id", "total_gasto_combinado", "total_fraudes_combinado").show(10)

# Detectar usuários suspeitos (mais de 5 fraudes)
usuarios_suspeitos = serving_view.filter(col("total_fraudes_combinado") > 5)
print(f"\nUsuários suspeitos detectados: {usuarios_suspeitos.count()}")

# (Análise do Lab) Você acabou de implementar uma Lambda Architecture em miniatura.
# Em produção, a Batch Layer rodaria no Hadoop/Spark a cada 1 hora, e a Speed Layer rodaria no Kafka + Flink em tempo real.

Serving View (Batch + Real-time):
+-------+---------------------+-----------------------+
|user_id|total_gasto_combinado|total_fraudes_combinado|
+-------+---------------------+-----------------------+
|      0|                    0|                   1001|
|      1|                 1001|                      0|
|      2|                 2002|                      0|
|      3|                 3003|                      0|
|      4|                 4004|                      0|
|      5|                 5005|                      1|
|      6|                 6006|                      0|
|      7|                 7007|                      0|
|      8|                 8008|                      0|
|      9|                 9009|                      0|
+-------+---------------------+-----------------------+
only showing top 10 rows


Usuários suspeitos detectados: 100



## Resumo do Laboratório: Lambda Architecture

Neste exercício, simulamos os três componentes principais de uma **Lambda Architecture**:

1.  **Batch Layer (Camada de Lote):** Processou um grande volume de dados históricos (1 milhão de registros) para criar uma visão consolidada do comportamento dos usuários.
2.  **Speed Layer (Camada de Velocidade):** Processou dados recentes em 'tempo real' (simulados), permitindo capturar mudanças rápidas de comportamento que os dados históricos ainda não refletiam.
3.  **Serving Layer (Camada de Serviço):** Unificou as visões de ambas as camadas para fornecer uma resposta completa e atualizada, permitindo a detecção de fraudes com base no histórico completo + eventos recentes.

Essa abordagem resolve o problema de latência em sistemas de Big Data, garantindo que as decisões sejam tomadas com a maior precisão possível.
